In [ ]:
#Imports & Environment Setup
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
# openrouter_key = os.getenv("OPENROUTER_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

if not groq_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add GROQ_API_KEY to .env"
    )

# Load configuration
from context_engineering.config import (
    VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
    CHAT_MODEL, EMBEDDING_MODEL, PROVIDER
)

provider = "GROQ"
print("Environment loaded")
print(f" Provider: {provider}")
print(f"Project root: {project_root}")

In [ ]:
#Import Chat Services
from context_engineering.application.chat_service.rag_service import RAGService;

print(" Chat services loaded from service layer")

 Chat services loaded from service layer


In [ ]:
#Connect to Vector Store & Initialize LLM
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from context_engineering.infrastructure.llm_providers import (
    get_default_embeddings,
    get_chat_llm
)

# Initialize using service factories (supports OpenRouter + multi-provider)
llm = get_chat_llm(temperature=0)

# Embeddings require an OpenAI-compatible provider (not Groq)
embedding_provider = "openai" if os.getenv("OPENAI_API_KEY") else "openrouter"
embeddings = get_default_embeddings(provider=embedding_provider)

print(f"LLM initialized: {CHAT_MODEL}")
print(f"Embeddings initialized: {EMBEDDING_MODEL} ({embedding_provider})")
print(f"Provider: {PROVIDER}")

# Connect to Qdrant local store
qdrant_path = VECTOR_DIR / "qdrant"
if not qdrant_path.exists():
    raise FileNotFoundError(" Run 02_chunking.ipynb first")

qdrant_client = QdrantClient(path=str(qdrant_path))
# qdrant-client 1.10+ renamed search() -> search_points()
if not hasattr(qdrant_client, "search") and hasattr(qdrant_client, "search_points"):
    qdrant_client.search = qdrant_client.search_points
vectorstore = Qdrant(
    client=qdrant_client,
    collection_name="primelands",
    embeddings=embeddings
 )

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RESULTS}
 )

print("✅ Connected to vector store")
print(f"📊 Collection size: {vectorstore.client.count(collection_name='sliding').count} vectors")

LLM initialized: llama-3.1-8b-instant
Embeddings initialized: openai/text-embedding-3-large (openrouter)
Provider: groq
✅ Connected to vector store
📊 Collection size: 1160 vectors


C:\Users\viraj\AppData\Local\Temp\ipykernel_18532\3319265090.py:29: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use `QdrantVectorStore` instead.
  vectorstore = Qdrant(


In [ ]:
# Initialize RAG Service
rag_service = RAGService(
    retriever=retriever,
    llm=llm,
    k=TOP_K_RESULTS
)

print("RAGService initialized")
print(f" Retrieval: top-{TOP_K_RESULTS} documents")

RAGService initialized
 Retrieval: top-4 documents


In [ ]:
# Generate Answer with RAG Service
USER_QUERY = "Tell me about properties in Colombo."

print(f"Query: {USER_QUERY}\n")
print("=" * 80)
print("GENERATING ANSWER WITH RAG SERVICE...")
print("=" * 80)

result = rag_service.generate(USER_QUERY)

print(f"\n⏱  Generation time: {result['generation_time']:.2f}s")
print(f" Documents used: {result['num_docs']}")
print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(result['answer'])
print("\n" + "=" * 80)
print("EVIDENCE URLS")
print("=" * 80)
for url in result['evidence_urls']:
    print(f"  - {url}")

Query: Tell me about properties in Colombo.

GENERATING ANSWER WITH RAG SERVICE...


AttributeError: 'QdrantClient' object has no attribute 'search'